# Messages

## Build a conversation from message objects and invoke the model.

In [12]:
import os
from urllib import response
from pydantic import SecretStr

from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

conversation = [
    SystemMessage("You translate English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications."),
]

response = model.invoke(conversation)
print(response.text)

J'aime construire des applications.


## Inspect an AIMessage: its text, tool calls, and token usage.

In [6]:
print(type(response))
print(response.text)
print(response.tool_calls)
print(response.usage_metadata)

<class 'langchain_core.messages.ai.AIMessage'>
J'aime créer des applications.
[]
{'input_tokens': 108, 'output_tokens': 74, 'total_tokens': 182, 'input_token_details': {'cache_read': 64}, 'output_token_details': {'reasoning': 58}}


## Answer a tool call by appending a ToolMessage keyed to the call id.

In [13]:
from langchain.messages import ToolMessage

tool_message = ToolMessage(
    content="Sunny, 72F",
    tool_call_id="call_123",
    name="get_weather",
)

conversation.append(tool_message)
print(conversation)
response = model.invoke(conversation)
print(response.text)


[SystemMessage(content='You translate English to French.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Translate: I love programming.', additional_kwargs={}, response_metadata={}), AIMessage(content="J'adore la programmation.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Translate: I love building applications.', additional_kwargs={}, response_metadata={}), ToolMessage(content='Sunny, 72F', name='get_weather', tool_call_id='call_123')]
J'adore créer des applications.


## Multimodal input uses content blocks inside a message dict.

In [27]:
from dotenv import load_dotenv

load_dotenv()
vision_model = init_chat_model(
    model_provider="openai",
    model=os.environ["FLEXAI_VISION_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)


message = {
        "role": "user",
        "content": [
            {"type": "text", "text": "What's in this image?"},
            {"type": "image_url", "image_url": {"url": "https://mamp.one/wp-content/uploads/2024/09/image-resources2.jpg"}},
        ],
    }

response = vision_model.invoke([message])
print(response.content_blocks)

[{'type': 'text', 'text': 'This image features a close-up, stylized portrait of a tabby cat. The cat is wearing pink-rimmed sunglasses and a bright pink jacket, giving it a "cool" or "retro-futuristic" aesthetic.\n\nThe lighting is heavily stylized using a "cyberpunk" or "synthwave" color palette, dominated by vibrant neon pink and cyan (blue) tones. This high-contrast lighting creates deep shadows and bright, glowing highlights across the cat\'s fur and clothing.'}]
